In [1]:
%matplotlib tk
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

root_path = "/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto"

ACTION_DIR = Path(root_path) / "action"
MAX_FILES = None
ARM_JOINT_COUNT = 6
MAX_STEP_POINTS = 30000
TOP_K_OUTLIERS = 15

action_files = sorted(ACTION_DIR.glob("*.json"))
if MAX_FILES is not None:
    action_files = action_files[:MAX_FILES]

assert action_files, f"No action json files found in: {ACTION_DIR}"

def pca_2d(x):
    x = np.asarray(x, dtype=np.float64)
    x_centered = x - x.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(x_centered, full_matrices=False)
    return x_centered @ vt[:2].T

def robust_zscore(x):
    x = np.asarray(x, dtype=np.float64)
    med = np.median(x, axis=0, keepdims=True)
    mad = np.median(np.abs(x - med), axis=0, keepdims=True)
    mad = np.where(mad < 1e-9, 1.0, mad)
    return 0.6745 * (x - med) / mad

episode_features = []
episode_files = []
step_points = []
step_point_files = []
joint_names = None

for action_file in tqdm(action_files, desc="Processing action files"):
    with open(action_file, "r") as f:
        data = json.load(f)
    if not data:
        continue

    joints = np.asarray([frame["robot"]["joint_positions"] for frame in data], dtype=np.float64)
    if joints.ndim != 2:
        continue

    if joint_names is None:
        joint_names = data[0]["robot"].get("joint_names", [f"joint_{i}" for i in range(joints.shape[1])])

    use_dim = min(ARM_JOINT_COUNT, joints.shape[1])
    joints = joints[:, :use_dim]
    diffs = np.diff(joints, axis=0) if len(joints) > 1 else np.zeros((1, use_dim), dtype=np.float64)
    abs_diffs = np.abs(diffs)

    feature = np.concatenate([
        joints.mean(axis=0),
        joints.std(axis=0),
        joints.min(axis=0),
        joints.max(axis=0),
        abs_diffs.mean(axis=0),
        abs_diffs.std(axis=0),
        abs_diffs.max(axis=0),
        np.array([len(joints)], dtype=np.float64),
    ])
    episode_features.append(feature)
    episode_files.append(action_file.name)

    step_points.append(joints)
    step_point_files.extend([action_file.name] * len(joints))

episode_features = np.asarray(episode_features, dtype=np.float64)
step_points = np.concatenate(step_points, axis=0)
step_point_files = np.asarray(step_point_files)

step_embed = pca_2d(step_points)
episode_feature_z = robust_zscore(episode_features)
episode_scores = np.sqrt((episode_feature_z ** 2).sum(axis=1))
episode_embed = pca_2d(episode_feature_z)

score_threshold = np.percentile(episode_scores, 97.5)
outlier_mask = episode_scores >= score_threshold
top_idx = np.argsort(episode_scores)[-TOP_K_OUTLIERS:][::-1]

rng = np.random.default_rng(42)
if len(step_embed) > MAX_STEP_POINTS:
    sample_idx = rng.choice(len(step_embed), size=MAX_STEP_POINTS, replace=False)
else:
    sample_idx = np.arange(len(step_embed))

outlier_files = np.asarray(episode_files)[outlier_mask]
step_colors = np.where(np.isin(step_point_files[sample_idx], outlier_files), "tab:red", "tab:blue")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].scatter(step_embed[sample_idx, 0], step_embed[sample_idx, 1], s=8, c=step_colors, alpha=0.5, linewidths=0)
for point, episode_name in zip(step_embed[sample_idx], step_point_files[sample_idx]):
    axes[0].annotate(episode_name.replace(".json", ""), (point[0], point[1]), fontsize=6, alpha=0.9)
axes[0].set_title(f"Step-level joint distribution (PCA 2D, first {use_dim} joints)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(episode_embed[:, 0], episode_embed[:, 1], s=28, c="lightgray", alpha=0.8, label="episodes")
axes[1].scatter(episode_embed[outlier_mask, 0], episode_embed[outlier_mask, 1], s=40, c="tab:red", alpha=0.95, label="outliers")
for idx in top_idx:
    axes[1].annotate(episode_files[idx], (episode_embed[idx, 0], episode_embed[idx, 1]), fontsize=8, alpha=0.9)
axes[1].set_title("Episode-level outlier view")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"analyzed episodes: {len(episode_files)}")
print(f"used joints: {joint_names[:use_dim]}")
print(f"outlier threshold (97.5 percentile): {score_threshold:.3f}")
print()
all_outlier_idx = np.where(outlier_mask)[0]
all_outlier_idx = all_outlier_idx[np.argsort(episode_scores[all_outlier_idx])[::-1]]
print(f"all outlier files: {len(all_outlier_idx)}")
for rank, idx in enumerate(all_outlier_idx, start=1):
    print(f"{rank:02d}. {episode_files[idx]} | score={episode_scores[idx]:.3f} | steps={int(episode_features[idx, -1])}")


Processing action files: 100%|██████████| 2248/2248 [01:19<00:00, 28.31it/s]


analyzed episodes: 2248
used joints: ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6']
outlier threshold (97.5 percentile): 53.444

all outlier files: 57
01. 1480.json | score=60.612 | steps=880
02. 2091.json | score=60.188 | steps=898
03. 1416.json | score=59.694 | steps=912
04. 1502.json | score=59.405 | steps=835
05. 1523.json | score=59.178 | steps=880
06. 1438.json | score=59.122 | steps=854
07. 1251.json | score=58.871 | steps=872
08. 2123.json | score=58.570 | steps=804
09. 1831.json | score=58.519 | steps=814
10. 0479.json | score=58.064 | steps=835
11. 1031.json | score=58.042 | steps=821
12. 1239.json | score=57.930 | steps=832
13. 2192.json | score=57.902 | steps=917
14. 0806.json | score=57.865 | steps=770
15. 0016.json | score=57.611 | steps=992
16. 1287.json | score=57.405 | steps=873
17. 0556.json | score=57.390 | steps=960
18. 0976.json | score=57.358 | steps=771
19. 0182.json | score=57.215 | steps=950
20. 1972.json | score=57.083 | steps=873
21. 1452.json |

# Warning!!!! Remove Files

In [ ]:
# rgb_path = Path(root_path) / "rgb"
# action_path = Path(root_path) / "action"
# import os
# import shutil
# for rank, idx in enumerate(all_outlier_idx, start=1):
#     print(f"{rank:02d}. {episode_files[idx]} | score={episode_scores[idx]:.3f} | steps={int(episode_features[idx, -1])}")
#     try:
#         os.remove(action_path / episode_files[idx])
#     except Exception as e:
#         print(f"Error removing files for {episode_files[idx]}: {e}")

#     try:
#         shutil.rmtree(rgb_path / episode_files[idx].replace(".json", ""))
#     except Exception as e:
#         print(f"Error removing folder for {episode_files[idx]}: {e}")

01. 0265.json | score=4195.960 | steps=1496
Error removing files for 0265.json: [Errno 2] No such file or directory: '/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto/action/0265.json'
Error removing folder for 0265.json: [Errno 2] No such file or directory: '/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto/rgb/0265'
02. 0819.json | score=1516.024 | steps=1390
Error removing files for 0819.json: [Errno 2] No such file or directory: '/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto/action/0819.json'
Error removing folder for 0819.json: [Errno 2] No such file or directory: '/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto/rgb/0819'
03. 2227.json | score=933.795 | steps=1175
Error removing files for 2227.json: [Errno 2] No such file or directory: '/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto/action/2227.json'
Error removing folder for 2227.json: [Errno 2] No such file or directory: '/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto/rgb/2227'
04. 0926.json | scor

In [4]:

from pathlib import Path


root_path = "/nas/Dataset/VLA/UON/Isaacsim_OMY_apple_picking_auto"
ACTION_DIR = Path(root_path) / "action"
action_list = sorted(ACTION_DIR.glob("*.json"))
action_list = [f.name for f in action_list]

for i in range(2000):
    if f"{i:04d}.json" not in action_list:
        print(i, end=" ")


40 73 79 85 163 265 277 325 358 376 388 391 444 475 496 505 519 539 540 553 628 658 709 794 819 926 982 1073 1181 1210 1218 1255 1276 1377 1564 1616 1840 1855 1903 1905 1942 

In [ ]:
a = 1